# LC 102 — Binary Tree Level Order Traversal
**Day 47 | Pattern: BFS Level-by-Level | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Snapshot the queue size before
processing each level. Pop exactly that many nodes, collect
their values, then push their children. Repeat.
</div>

## Official Problem Statement

Given the `root` of a binary tree, return the level order traversal
of its nodes' values (i.e., from left to right, level by level).

**Constraints:**
- Number of nodes: `[0, 2000]`
- `-1000 <= Node.val <= 1000`

## What This Is Actually Asking

Group all nodes on the same depth into one list.
Return a list of lists, one inner list per depth level.
Order within each level is left-to-right.
An empty tree returns an empty list.
This is BFS but with an extra grouping step per level.

## Walk Through an Example by Hand

```
Tree:    3
        / \
       9  20
          / \
         15   7
```
- Start: queue=[3]
- Level 0: size=1, pop 3, push 9,20. level=[3]
- Level 1: size=2, pop 9 (no children), pop 20 (push 15,7).
  level=[9,20]
- Level 2: size=2, pop 15,7 (no children). level=[15,7]
- Result: [[3],[9,20],[15,7]]

## The Picture

```
Queue state (BFS with level snapshots):

Level 0:  [3]            size=1  -> collect [3]
           |
Level 1:  [9, 20]        size=2  -> collect [9, 20]
               |
Level 2:  [15, 7]        size=2  -> collect [15, 7]

Key trick:
  level_size = len(queue)   <- snapshot BEFORE the inner loop
  for _ in range(level_size): <- pop exactly that many
      node = queue.popleft()
```

## When To Use This Pattern

- When a tree problem asks for results **grouped by depth**,
  think BFS with level-size snapshot.
- When you need the **shortest path** in an unweighted graph,
  think BFS (same pattern).
- When you need to process nodes **width-first** (all siblings
  before children), think BFS with a deque.
- When the problem mentions "level" or "layer", think
  `level_size = len(queue)` trick.

## The Approach

Use a deque initialized with the root. At the start of each
iteration, snapshot the queue length as `level_size`. Pop
exactly `level_size` nodes, collect their values, and enqueue
their non-null children. Append the level list to the result.
Repeat until the queue is empty.

In [ ]:
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

In [ ]:
def test_harness(func):
    cases = [
        ([3,9,20,None,None,15,7], [[3],[9,20],[15,7]]),
        ([1],                     [[1]]),
        ([],                      []),
        ([1,2,3,4,5],             [[1],[2,3],[4,5]]),
        ([1,None,2,None,3],       [[1],[2],[3]]),
    ]
    passed = 0
    for i, (vals, expected) in enumerate(cases):
        root = make_tree(vals)
        result = func(root)
        # Sort inner lists for robust comparison
        ok = (
            [sorted(lvl) for lvl in result]
            == [sorted(lvl) for lvl in expected]
        )
        status = 'PASSED' if ok else 'FAILED'
        if ok:
            passed += 1
        print(f'Case {i+1}: {status} | '
              f'Expected {expected} | Got {result}')
    print(f'\n{passed}/{len(cases)} passed')

In [ ]:
def levelOrder(
    root: Optional[TreeNode]
) -> List[List[int]]:
    """
    Return level order traversal as list of lists.

    Strategy: BFS with deque. Snapshot queue length at the
    start of each level; pop exactly that many nodes to
    form one level's result.

    Args:
        root: Root of the binary tree.
    Returns:
        List of lists, one per level, left to right.
    """
    # print(f'Root: {root.val if root else None}')
    result = []
    if not root:
        return result
    # queue = deque([root])
    # while queue:
    #     level_size = len(queue)
    #     level = []
    #     for _ in range(level_size):
    #         node = queue.popleft()
    #         level.append(node.val)
    #         if node.left: queue.append(node.left)
    #         if node.right: queue.append(node.right)
    #     result.append(level)
    # print(f'Result: {result}')
    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(levelOrder)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| DFS with depth tracking | O(n) | O(n) | Recursion stack |
| BFS level snapshot | O(n) | O(n) | Queue max = widest level |
| BFS (optimal) | O(n) | O(w) | w = max tree width |

## Real World Connection

At **AWS**, the CloudFormation stack dependency resolution
processes resources level by level — resources with no
dependencies first, then those depending on them.
At **Citi**, trade settlement hierarchies (parent trades and
their child allocations) are processed depth-by-depth.
In **data pipelines**, DAG task schedulers like Apache Airflow
emit tasks in topological levels — this BFS grouping is the
core scheduling algorithm under the hood.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra